In [1]:
import kagglehub
import pathlib
import shutil

In [2]:
# Download latest version
path = pathlib.Path(kagglehub.dataset_download("muratkokludataset/rice-image-dataset"))

DATA_PATH = pathlib.Path("./data")

if DATA_PATH.exists():
    shutil.rmtree(DATA_PATH)

shutil.copytree(path, DATA_PATH)

Using Colab cache for faster access to the 'rice-image-dataset' dataset.


PosixPath('data')

In [3]:
!pip install split-folders

In [4]:
import splitfolders
import tensorflow as tf
import os

if os.path.exists("./data/Rice_Image_Dataset/Rice_Citation_Request.txt"):
    os.remove("./data/Rice_Image_Dataset/Rice_Citation_Request.txt")

DATASET_PATH = pathlib.Path("./data/Rice_Image_Dataset")

splitfolders.ratio(DATASET_PATH, output="SPLITTED", seed=42, ratio=(.8, 0.1,0.1))

Copying files: 75000 files [00:11, 6537.55 files/s]


In [5]:
train_dir = pathlib.Path("./SPLITTED/train")
test_dir = pathlib.Path("./SPLITTED/test")
val_dir = pathlib.Path("./SPLITTED/val")

img_size = (256,256)
batch_size = 32
seed = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    image_size=img_size,
    batch_size=batch_size,
    seed = seed
    )

val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=img_size,
    batch_size=batch_size,
    seed = seed
    )

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size,
    seed = seed
    )

train_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

Found 60000 files belonging to 5 classes.
Found 7500 files belonging to 5 classes.
Found 7500 files belonging to 5 classes.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

## Transfer Learning

In [6]:
backbone = tf.keras.applications.EfficientNetB5(
    include_top=False,
    weights = 'imagenet',
    input_shape = img_size + (3,)
    )


115263384/115263384 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [7]:
backbone.trainable = False

In [8]:
preprocess_input = tf.keras.applications.efficientnet.preprocess_input

In [9]:
classification_heads = tf.keras.Sequential([
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(5, activation='softmax')
])

In [10]:
inputs = tf.keras.layers.Input(shape=img_size + (3,))
x = preprocess_input(inputs)
x = backbone(x)
outputs = classification_heads(x)
model = tf.keras.Model(inputs, outputs)

In [12]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

learning_rate_adapter = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    patience=2,
    factor = 0.2,
)

checkpoint_model = tf.keras.callbacks.ModelCheckpoint(
    filepath = './best_model.keras',
    monitor='val_loss',
    save_best_only=True,
)

checkpoint_weights = tf.keras.callbacks.ModelCheckpoint(
    filepath = './best.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True
)

In [13]:
model.compile(
    tf.keras.optimizers.AdamW(learning_rate=0.0001),
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy'],
    )

In [14]:
history = model.fit(
    train_dataset,
    validation_data = val_dataset,
    epochs = 50,
    callbacks = [
        early_stopping,
        learning_rate_adapter,
        checkpoint_model,
        checkpoint_weights
    ]
)

Epoch 1/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 446s 202ms/step - accuracy: 0.9331 - loss: 0.3313 - val_accuracy: 0.9764 - val_loss: 0.1152 - learning_rate: 1.0000e-04
Epoch 2/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 369s 186ms/step - accuracy: 0.9743 - loss: 0.1072 - val_accuracy: 0.9845 - val_loss: 0.0693 - learning_rate: 1.0000e-04
Epoch 3/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 379s 185ms/step - accuracy: 0.9812 - loss: 0.0749 - val_accuracy: 0.9880 - val_loss: 0.0517 - learning_rate: 1.0000e-04
Epoch 4/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 347s 185ms/step - accuracy: 0.9850 - loss: 0.0592 - val_accuracy: 0.9907 - val_loss: 0.0422 - learning_rate: 1.0000e-04
Epoch 5/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 387s 187ms/step - accuracy: 0.9864 - loss: 0.0511 - val_accuracy: 0.9921 - val_loss: 0.0367 - learning_rate: 1.0000e-04
Epoch 6/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 353s 188ms/step - accuracy: 0.9876 - loss: 0.0449 - val_accuracy: 0.9927 - val_loss: 0.0328 - learning_rate: 1.0000e-04
Epoch 7/50
1875/1875 ━━━━━━━

KeyboardInterrupt: 

## 99.57% di validation_accuracy all'epoca 26 (potrebbe continuare)

In [15]:
best_model = tf.keras.models.load_model("./best_model.keras")

model.load_weights("./best.weights.h5")

In [16]:
results = model.evaluate(test_dataset)
print("Test Loss:", results[0])
print("Test Accuracy:", results[1])

235/235 ━━━━━━━━━━━━━━━━━━━━ 38s 160ms/step - accuracy: 0.9961 - loss: 0.0129
Test Loss: 0.01286225114017725
Test Accuracy: 0.9961333274841309


## 99.61% di accuracy sul test dataset